In [0]:
# Check the built-in sample sales data
display(spark.sql("SHOW TABLES IN samples.tpch"))

orders = spark.table("samples.tpch.orders")
print("Orders:", orders.count())
display(orders.limit(5))

database,tableName,isTemporary
tpch,customer,false
tpch,lineitem,false
tpch,nation,false
tpch,orders,false
tpch,part,false
tpch,partsupp,false
tpch,region,false
tpch,supplier,false


Orders: 7500000


o_orderkey,o_custkey,o_orderstatus,o_totalprice,o_orderdate,o_orderpriority,o_clerk,o_shippriority,o_comment
11396166,179329,O,187683.10,1996-06-22,4-NOT SPECIFIED,Clerk#000004689,0,"ep fluffily regular packages. regular, final courts ag"
11396167,473245,F,117554.58,1994-07-20,1-URGENT,Clerk#000001856,0,efully after the carefully express packages; final r
11396192,77549,O,47611.56,1996-01-03,3-MEDIUM,Clerk#000001450,0,es. regular tithes poach careful
11396193,610651,F,121869.26,1994-11-29,5-LOW,Clerk#000003696,0,"are furiously along the bold, even ideas. even, final pinto beans"
11396194,193169,O,241066.47,1998-03-21,4-NOT SPECIFIED,Clerk#000002476,0,lar instructions use slyly carefully furious instructions. quickly


In [0]:
from pyspark.sql import functions as F

# Create a schema (database) for the project
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.sales_lakehouse")
spark.sql("USE workspace.sales_lakehouse")

# BRONZE: copy raw source tables as-is + add ingestion metadata
sources = ["orders", "customer", "nation", "region"]

for t in sources:
    df = (spark.table(f"samples.tpch.{t}")
          .withColumn("_ingested_at", F.current_timestamp())
          .withColumn("_source", F.lit(f"samples.tpch.{t}")))
    df.write.format("delta").mode("overwrite").saveAsTable(f"bronze_{t}")
    print(f"bronze_{t}: {spark.table(f'bronze_{t}').count():,} rows")

bronze_orders: 7,500,000 rows
bronze_customer: 750,000 rows
bronze_nation: 25 rows
bronze_region: 5 rows


In [0]:
from pyspark.sql import functions as F

# ---------- SILVER ORDERS: clean, validate, deduplicate ----------
raw = spark.table("bronze_orders")

orders = (raw
    .select(
        F.col("o_orderkey").alias("order_id"),
        F.col("o_custkey").alias("customer_id"),
        F.col("o_orderstatus").alias("status"),
        F.col("o_totalprice").cast("double").alias("total_price"),
        F.col("o_orderdate").alias("order_date"),
        F.trim(F.col("o_orderpriority")).alias("priority"))
    .dropDuplicates(["order_id"]))

# Data quality rules
rules = {
    "order_id_not_null":   F.col("order_id").isNotNull(),
    "customer_id_not_null": F.col("customer_id").isNotNull(),
    "price_positive":      F.col("total_price") > 0,
    "valid_status":        F.col("status").isin("O", "F", "P"),
    "date_not_null":       F.col("order_date").isNotNull(),
}
is_valid = F.lit(True)
for cond in rules.values():
    is_valid = is_valid & cond

valid   = orders.filter(is_valid)
invalid = orders.filter(~is_valid)

valid.write.format("delta").mode("overwrite").saveAsTable("silver_orders")
invalid.write.format("delta").mode("overwrite").saveAsTable("quarantine_orders")

# ---------- SILVER CUSTOMERS: enrich with nation + region ----------
cust = (spark.table("bronze_customer").alias("c")
    .join(spark.table("bronze_nation").alias("n"), F.col("c.c_nationkey") == F.col("n.n_nationkey"))
    .join(spark.table("bronze_region").alias("r"), F.col("n.n_regionkey") == F.col("r.r_regionkey"))
    .select(
        F.col("c.c_custkey").alias("customer_id"),
        F.col("c.c_name").alias("customer_name"),
        F.col("c.c_mktsegment").alias("market_segment"),
        F.col("c.c_acctbal").cast("double").alias("account_balance"),
        F.col("n.n_name").alias("country"),
        F.col("r.r_name").alias("region"))
    .dropDuplicates(["customer_id"]))
cust.write.format("delta").mode("overwrite").saveAsTable("silver_customers")

# ---------- Data quality report ----------
total = raw.count()
print(f"Bronze orders     : {total:,}")
print(f"Silver orders     : {spark.table('silver_orders').count():,}")
print(f"Quarantined orders: {spark.table('quarantine_orders').count():,}")
print(f"Silver customers  : {spark.table('silver_customers').count():,}")
for name, cond in rules.items():
    fails = orders.filter(~cond).count()
    print(f"  rule {name:22s}: {fails} failures")

Bronze orders     : 7,500,000
Silver orders     : 7,500,000
Quarantined orders: 0
Silver customers  : 750,000
  rule order_id_not_null     : 0 failures
  rule customer_id_not_null  : 0 failures
  rule price_positive        : 0 failures
  rule valid_status          : 0 failures
  rule date_not_null         : 0 failures


In [0]:
from pyspark.sql import functions as F

orders = spark.table("silver_orders")
cust   = spark.table("silver_customers")
fact   = orders.join(cust, "customer_id")

# GOLD 1: monthly revenue by region
gold_region_month = (fact
    .withColumn("month", F.date_trunc("month", "order_date"))
    .groupBy("region", "month")
    .agg(F.round(F.sum("total_price"), 2).alias("revenue"),
         F.count("order_id").alias("orders"),
         F.round(F.avg("total_price"), 2).alias("avg_order_value")))
gold_region_month.write.format("delta").mode("overwrite").saveAsTable("gold_revenue_by_region_month")

# GOLD 2: market segment performance
gold_segment = (fact.groupBy("market_segment")
    .agg(F.round(F.sum("total_price"), 2).alias("revenue"),
         F.countDistinct("customer_id").alias("customers"),
         F.round(F.avg("total_price"), 2).alias("avg_order_value"))
    .orderBy(F.desc("revenue")))
gold_segment.write.format("delta").mode("overwrite").saveAsTable("gold_segment_performance")

# GOLD 3: top 100 customers by lifetime value
gold_top = (fact.groupBy("customer_id", "customer_name", "country", "market_segment")
    .agg(F.round(F.sum("total_price"), 2).alias("lifetime_value"),
         F.count("order_id").alias("orders"))
    .orderBy(F.desc("lifetime_value")).limit(100))
gold_top.write.format("delta").mode("overwrite").saveAsTable("gold_top_customers")

print("Gold tables created.")
display(spark.table("gold_segment_performance"))
display(spark.table("gold_revenue_by_region_month")
        .groupBy("region").agg(F.round(F.sum("revenue")/1e9, 2).alias("revenue_bn"))
        .orderBy(F.desc("revenue_bn")))

Gold tables created.


market_segment,revenue,customers,avg_order_value
FURNITURE,2.2776680807412E11,100394,151162.89
BUILDING,2.274974287302E11,100270,151145.72
MACHINERY,2.2656302185645E11,99988,151127.45
AUTOMOBILE,2.2631118915613E11,99973,151077.38
HOUSEHOLD,2.2530076742935E11,99364,151112.32


region,revenue_bn
EUROPE,227.71
MIDDLE EAST,227.52
ASIA,227.18
AMERICA,225.67
AFRICA,225.35


In [0]:
# 1. Delta version history (audit trail of every write)
display(spark.sql("DESCRIBE HISTORY silver_orders")
        .select("version", "timestamp", "operation"))

# 2. Time travel: read an earlier version of a table
v0 = spark.sql("SELECT COUNT(*) AS rows FROM silver_orders VERSION AS OF 0")
display(v0)

# 3. Optimise the large table for faster queries
display(spark.sql("OPTIMIZE silver_orders"))

# 4. Document the tables (Unity Catalog comments)
spark.sql("COMMENT ON TABLE silver_orders IS 'Cleaned, validated and deduplicated orders (Silver layer)'")
spark.sql("COMMENT ON TABLE gold_revenue_by_region_month IS 'Monthly revenue, orders and AOV by region (Gold layer)'")

display(spark.sql("SHOW TABLES IN workspace.sales_lakehouse"))

version,timestamp,operation
0,2026-10-21T22:58:59.000Z,CREATE OR REPLACE TABLE AS SELECT


rows
7500000


path,metrics
,"List(1, 2, List(60034686, 60034686, 6.0034686E7, 1, 60034686), List(30038639, 30054350, 3.00464945E7, 2, 60092989), 0, null, null, 0, 1, 3, 1, true, 0, 0, 1792623650528, 1792623655409, 8, 1, null, List(0, 0), null, 6, 6, 1894, 0, null, null, 0)"


database,tableName,isTemporary
sales_lakehouse,bronze_customer,false
sales_lakehouse,bronze_nation,false
sales_lakehouse,bronze_orders,false
sales_lakehouse,bronze_region,false
sales_lakehouse,gold_revenue_by_region_month,false
sales_lakehouse,gold_segment_performance,false
sales_lakehouse,gold_top_customers,false
sales_lakehouse,quarantine_orders,false
sales_lakehouse,silver_customers,false
sales_lakehouse,silver_orders,false
